<a href="https://colab.research.google.com/github/parthsharma5575/Ai-Ml-GenAi/blob/main/AI_Agents_with_LangChain_%26_Groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  AI Agents with LangChain & Groq
### Practical Implementation: Tools, Wikipedia, Tavily Web Search & Function Calling
---

##  What is an AI Agent?

An **AI Agent** is a Large Language Model (LLM) that can:
-  **Think** — reason about what needs to be done
-  **Use Tools** — call functions like search, calculator, APIs
-  **Observe** — look at tool results and decide next steps
-  **Repeat** — keep going until the task is done
```
User Question
     │
     ▼
  [THINK]   ← LLM decides what tool to call
     │
     ▼
  [CALL TOOL] ← Wikipedia / Tavily / add / multiply
     │
     ▼
  [OBSERVE]  ← Read tool result
     │
     ▼
 Done? ──No──► [THINK] again
     │Yes
     ▼
 Final Answer
```

##  Tools We'll Use in This Notebook

| Tool | Purpose |
|---|---|
| `WikipediaQueryRun` | Search Wikipedia for factual info |
| `TavilySearch` | Search the live web for recent news |
| `add` | Custom tool: adds two numbers |
| `multiply` | Custom tool: multiplies two numbers |
| `ChatGroq` | Fast, free LLM via Groq API (LLaMA 3.3) |

---

In [ ]:
!pip install -q langchain_community wikipedia langchain_tavily langchain_groq

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


##  Step 2: Set Up API Keys

We need two API keys:
- **Groq API Key** → Free at https://console.groq.com  (fast LLaMA 3.3 model)
- **Tavily API Key** → Free at https://app.tavily.com  (real-time web search)

>  **Security Note:** Never hardcode API keys in notebooks you share publicly.
> Use `getpass` or environment variables to keep them safe.

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Groq API Key: ")
os.environ["TAVILY_API_KEY"] = getpass("Tavily API Key: ")

Groq API Key: ··········
Tavily API Key: ··········


## 🌐 Step 3: Set Up the Wikipedia Tool

`WikipediaQueryRun` lets the agent search Wikipedia for factual information.

**Parameters:**
- `top_k_results=1` → return only the top 1 article
- `doc_content_chars_max=200` → limit result to 200 characters (keeps it concise)

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

/tmp/ipykernel_4346/2961514931.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [ ]:
api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=200
)

wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

wiki_tool.invoke({"query": "langchain"})

'Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain'

##  Step 4: Set Up the Tavily Web Search Tool

`TavilySearch` gives the agent access to **real-time web search** —
great for news, recent events, or anything not in Wikipedia.

**Parameters:**
- `max_results=5` → fetch up to 5 web results
- `topic="general"` → general-purpose search (can also use "news")

In [ ]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results=5,
    topic="general"
)

result = tavily_tool.invoke({"query": "What is the latest news in AI?"})




In [ ]:
for r in result.get("results", [])[:4]:  # Show first 2 results
    print(f"  • {r['title']}")

  • AI News | Latest News | Insights Powering AI-Driven Business Growth
  • Artificial Intelligence - Latest AI News and Analysis
  • AI News: The AI Launch That Crashed The Market
  • AI News, Updates, Products and Reviews


##  Step 5: Create Custom Math Tools

We create two simple math tools using the `@tool` decorator from LangChain.

**Key concept:** The **docstring** (the text in `""" """`) is what the LLM
reads to understand when and how to use the tool. Write it clearly!

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """
    Adds two integers together.
    Use this when the user wants to add or sum two numbers.
    Example: add(3, 5) returns 8
    """
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """
    Multiplies two integers together.
    Use this when the user wants to multiply or find the product of two numbers.
    Example: multiply(4, 6) returns 24
    """
    return a * b


In [ ]:
add.invoke({"a": 10, "b": 25})

35

In [ ]:
multiply.invoke({"a": 6, "b": 7})

42

## Step 6: Bundle All Tools Together

Now we combine all 4 tools into a single list that we'll pass to the LLM.
```python
tools = [wiki_tool, add, multiply, tavily_tool]
```

The LLM will choose **which tool to call** based on the user's question.

In [ ]:
tools = [wiki_tool, add, multiply, tavily_tool]

##  Step 7: Initialize the LLM with Tool Binding

We use **Groq's LLaMA 3.3 70B** model — it's free and very fast.

**`bind_tools(tools)`** — this tells the LLM about all available tools
so it knows what it can call and how to call them.

**`temperature=0`** — keeps responses consistent and deterministic,
which is important for tool-calling agents.

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(

               model = 'llama-3.3-70b-versatile',
               temperature = 0
)

In [ ]:
llm_with_tool = llm.bind_tools(tools)

In [ ]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/usr/local/lib/python3.12/dist-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='add', description='Adds two integers together.\nUse this when the user wants to add or sum two numbers.\nExample: add(3, 5) returns 8', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x7df247011580>),
 StructuredTool(name='multiply', description='Multiplies two integers together.\nUse this when the user wants to multiply or find the product of two numbers.\nExample: multiply(4, 6) returns 24', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x7df24681f6a0>),
 TavilySearch(max_results=5, topic='general', api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))]

##  Step 8: Send a Multi-Task Question to the LLM

We send one question that requires **multiple tools** at the same time:

1. *"What is LangChain?"* → needs **Wikipedia**
2. *"What is 5 × 15?"* → needs **multiply**
3. *"Summarize recent AI news"* → needs **Tavily web search**

The LLM will decide to call all three tools simultaneously!

In [ ]:
from langchain_core.messages import HumanMessage


In [ ]:
user_question = "What is langchain and what is 5*15 and summarize the recent AI news? and waht is 5+5"


In [ ]:
messages = [HumanMessage(content=user_question)]

In [ ]:
print(user_question)

What is langchain and what is 5*15 and summarize the recent AI news?


In [ ]:
ai_msg = llm_with_tool.invoke(messages)
messages.append(ai_msg)
for i in ai_msg.tool_calls:
  print(i['name'],i['args'])

wikipedia {'query': 'LangChain'}
multiply {'a': 5, 'b': 15}
tavily_search {'query': 'recent AI news', 'topic': 'news'}


## Step 9: Execute the Tool Calls

The LLM told us which tools to call — now we actually **run those tools**
and collect their results.

This is the "Action → Observation" step in the agent loop:
- **Action** = calling the tool
- **Observation** = the tool's result

We store each result as a `ToolMessage` and append it to the conversation.

In [ ]:
tool_map = {
    "add":           add,
    "multiply":      multiply,
    "wikipedia":     wiki_tool,
    "tavily_search": tavily_tool,
}

In [ ]:
tool_map

{'add': StructuredTool(name='add', description='Adds two integers together.\nUse this when the user wants to add or sum two numbers.\nExample: add(3, 5) returns 8', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x7df247011580>),
 'multiply': StructuredTool(name='multiply', description='Multiplies two integers together.\nUse this when the user wants to multiply or find the product of two numbers.\nExample: multiply(4, 6) returns 24', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x7df24681f6a0>),
 'wikipedia': WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/usr/local/lib/python3.12/dist-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 'tavily_search': TavilySearch(max_results=5, topic='general', api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))}

In [ ]:
for tool_call in ai_msg.tool_calls:
    tool_name = tool_call["name"].lower()
    selected_tool = tool_map[tool_name]
    tool_result = selected_tool.invoke(tool_call)
    messages.append(tool_result)

    print(f"{tool_name} → done")


for m in messages:
    print(f"   {type(m).__name__}: {str(m.content)[:80]}...")

wikipedia → done
multiply → done
tavily_search → done
   HumanMessage: What is langchain and what is 5*15 and summarize the recent AI news?...
   AIMessage: ...
   AIMessage: ...
   AIMessage: ...
   ToolMessage: Page: LangChain
Summary: LangChain is a software framework that helps facilitate...
   ToolMessage: 75...
   ToolMessage: {"query": "recent AI news", "follow_up_questions": null, "answer": null, "images...


##  Step 10: Get the Final Answer

Now we pass the **full conversation history** (original question + all tool results)
back to the LLM. It reads everything and generates a clean, combined final answer.

This is the last step of the agent loop — the LLM synthesizes all observations
into one coherent response for the user.

In [ ]:
print('*'*50)

final_responce = llm_with_tool.invoke(messages)

print(f"Final Answer: {final_responce.content}")

**************************************************
Final Answer: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. 

The result of 5*15 is 75.

Recent AI news includes the launch of Gemini 3.5, a new AI model that delivers frontier intelligence for agents and coding, and the introduction of Gemini Omni, which combines the ability to reason with the ability to create. Additionally, there have been advancements in AI-powered tools for science, such as Gemini for Science, and the development of new AI models like Claude Sonnet 5 and Fable. Furthermore, companies like HP and OpenAI are working together to accelerate enterprise workflows with AI, and there have been significant investments in AI research and development.


##  Summary — What We Built

### Key Concepts Learned

| Concept | What it means |
|---|---|
| `@tool` | Decorator that turns a Python function into an agent tool |
| `bind_tools()` | Tells the LLM about available tools |
| `tool_calls` | List of tools the LLM decided to call |
| `ToolMessage` | The result returned after executing a tool |
| Multi-tool calling | LLM can call several tools in one step |

